# WattBot 2026: RAG pipeline

Answers questions about AI and data-center energy, water and carbon from 122 documents, with citations, and abstains
when the documents cannot support an answer.

1. **Documents**: fetch each PDF once (politely), cache it, cut pages into chunks
2. **Retrieval**: BM25 keywords + `bge-small` embeddings, reciprocal rank fusion; refused questions retry with more context
3. **Answering**: `databricks-gpt-oss-120b`, excerpts first then the question; optional majority vote over 3 samples
4. **Scoring**: the organisers' `Score.py` on the 245 training questions, then `submission.csv`

**Setup:** Settings → Internet **on**, Accelerator **GPU T4 x2**. Add-ons → Secrets: `DATABRICKS_HOST`, `DATABRICKS_TOKEN`,
`GITHUB_TOKEN` (read access to the private repo). Input: the **WattBot 2026** competition data, and after the first run
your private **corpus dataset** (see section 1).

In [ ]:
!pip install -q pdfplumber
import os, subprocess, sys
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
for name in ("DATABRICKS_HOST", "DATABRICKS_TOKEN"):
    os.environ[name] = secrets.get_secret(name)

# The repo is private while the competition runs (Kaggle rules on sharing competition code); clone with a token.
token = secrets.get_secret("GITHUB_TOKEN")
subprocess.run("rm -rf wattbot-2026", shell=True)
subprocess.run(["git", "clone", "-q", f"https://{token}@github.com/BensonAnalyst/wattbot-2026.git"], check=True)
print(subprocess.run(["git", "-C", "wattbot-2026", "log", "-1", "--format=%h %s (%cr)"],
                     capture_output=True, text=True).stdout)
sys.path.insert(0, "wattbot-2026/src")

In [ ]:
import glob, importlib.util, json
import pandas as pd

from wattbot.docs import build_chunks, download_pdfs, first_author, load_metadata
from wattbot.llm import make_databricks_llm
from wattbot.retrieve import Retriever, doc_recall
from wattbot.run import answer_all, make_embedder
from wattbot.score import parse_refs, score_frame

DATA = os.path.dirname(glob.glob("/kaggle/input/**/train_QA.csv", recursive=True)[0])
WORK = "/kaggle/working"
MODEL = "databricks-gpt-oss-120b"
K, K_RETRY = 8, 20   # excerpts per question; a refused question retries with 20
N_VOTES = 1          # 1 = single deterministic answer; 3 = majority vote (3x the LLM calls)

meta = load_metadata(f"{DATA}/metadata.csv")
train = pd.read_csv(f"{DATA}/train_QA.csv", keep_default_na=False)
test = pd.read_csv(f"{DATA}/test_Q.csv", keep_default_na=False)
urls = dict(zip(meta["id"], meta["url"]))
authors = {r.id: first_author(r.citation) for r in meta.itertuples()}

# The organisers' scorer, if it ships with the data; otherwise the tested local mirror.
score_files = glob.glob("/kaggle/input/**/Score.py", recursive=True)
official = None
if score_files:
    spec = importlib.util.spec_from_file_location("official_score", score_files[0])
    official = importlib.util.module_from_spec(spec); spec.loader.exec_module(official)
print(f"Data: {DATA}\n{len(meta)} documents, {len(train)} train, {len(test)} test questions; "
      f"scorer: {'official Score.py' if official else 'local mirror'}")

## 1. Documents (fetched once, then cached)
The rules ask entrants to fetch each document once and cache it. The first run downloads the PDFs slowly (a pause
between requests) into `/kaggle/working/pdfs`. **After the first run:** Output panel → `pdfs` folder → *New Dataset*,
name it `wattbot-2026-corpus`, keep it **private**, and add it as an Input. Later runs read it and download nothing.

In [ ]:
cache_dirs = sorted({os.path.dirname(p) for p in glob.glob("/kaggle/input/**/*.pdf", recursive=True)})
pdfs = download_pdfs(meta, f"{WORK}/pdfs", cache_dirs=cache_dirs, delay=3.0)
missing = [d for d, p in pdfs.items() if p is None]
print(f"Cached inputs: {cache_dirs or 'none'}")
print(f"PDFs available: {len(pdfs) - len(missing)} of {len(pdfs)}. Missing: {missing or 'none'}")

cached_chunks = glob.glob("/kaggle/input/**/chunks.json", recursive=True)
chunks = build_chunks(meta, pdfs, cache_path=cached_chunks[0] if cached_chunks else f"{WORK}/chunks.json")
per_doc = pd.Series([c["doc_id"] for c in chunks]).value_counts()
print(f"{len(chunks)} chunks from {per_doc.size} documents; fewest: {per_doc.tail(3).to_dict()}")

## 2. Retrieval check on the training questions (no LLM)

In [ ]:
embed = make_embedder()
retrievers = {"bm25": Retriever(chunks, authors), "hybrid": Retriever(chunks, authors, embed_fn=embed)}
answerable = train[train["answer_value"] != "is_blank"]
for name, r in retrievers.items():
    rec = [doc_recall(r.search(q.question, k=K), parse_refs(q.ref_id)) for q in answerable.itertuples()]
    by_type = {t: round(sum(x for x, f in zip(rec, answerable[t]) if str(f) == "1") /
                        max(1, sum(str(f) == "1" for f in answerable[t])), 3)
               for t in ("CrossPaper", "Reconcile", "Table", "Figure") if t in answerable}
    print(f"{name:<7} document recall@{K}: {sum(rec) / len(rec):.3f}   by type: {by_type}")

## 3. Answer the training questions and score them

In [ ]:
llm = make_databricks_llm(MODEL)
print("Model says:", llm([{"role": "user", "content": "Reply with the single word: ready"}]))

retriever = retrievers["hybrid"]
train_pred, train_ctx, train_retried = answer_all(train, retriever, llm, urls, f"{WORK}/llm_cache.json", MODEL,
                                                  k=K, k_retry=K_RETRY, n_votes=N_VOTES)
if official:
    train_score = official.score(train.copy(), train_pred.copy(), "id", verbose=True)
detail, mirror_score = score_frame(train_pred, train)
print(f"Train score: {train_score if official else mirror_score:.4f} (local mirror: {mirror_score})")
print(f"Abstained: {(train_pred['answer_value'] == 'is_blank').sum()} "
      f"(truly unanswerable: {(train['answer_value'] == 'is_blank').sum()}); retried wider: {len(train_retried)}")
flags = [c for c in ("Quote", "Table", "Figure", "Math", "CrossPaper", "Reconcile", "is_NA") if c in train]
rows = [(f, int((train[f].astype(str) == "1").sum()),
         round(detail.loc[(train[f].astype(str) == "1").values, "score"].mean(), 3)) for f in flags]
rows += [(f"Cohort {c}", int((train["Cohort"].astype(str) == c).sum()),
          round(detail.loc[(train["Cohort"].astype(str) == c).values, "score"].mean(), 3)) for c in ("2025", "2026")]
display(pd.DataFrame(rows, columns=["question type", "n", "mean score"]))

In [ ]:
# Error analysis: every question that lost points, worst first
pd.set_option("display.max_colwidth", 100)
misses = detail.assign(**{f: train[f].values for f in flags}).query("score < 1").sort_values("score")
display(misses[["id", "question", "truth", "pred", "truth_ref", "pred_ref", "score"] + flags].head(60))
train_pred.merge(detail[["id", "score"]], on="id").to_csv(f"{WORK}/train_predictions.csv", index=False)

## 4. Answer the test questions and write the submission

In [ ]:
sub, test_ctx, test_retried = answer_all(test, retriever, llm, urls, f"{WORK}/llm_cache.json", MODEL,
                                         k=K, k_retry=K_RETRY, n_votes=N_VOTES)
assert len(sub) == len(test) and sub["id"].is_unique and set(sub["id"]) == set(test["id"])
assert sub["explanation"].astype(str).str.strip().ne("").all()  # the scorer rejects empty explanations
sub.to_csv(f"{WORK}/submission.csv", index=False)
errors = sub["explanation"].astype(str).str.startswith("ERROR").sum()
print(f"submission.csv: {len(sub)} rows, {(sub['answer_value'] == 'is_blank').sum()} abstained, "
      f"{len(test_retried)} retried wider, {errors} failed calls")